In [4]:
import requests
from bs4 import BeautifulSoup
import duckdb
import os

from pathlib import Path

# ensuring that os.chdir is idempotent and that we are in the project root directory,
# not inside the notebooks directory
if 'notebooks' not in os.listdir(Path.cwd()):
    print("Still inside notebooks directory, changing to project root directory.")
    os.chdir(Path.cwd().parent)
    print("Current working directory after change: ", Path.cwd())
else:
    print(f"Already in parent directory (current working directory: {Path.cwd()})")

# load config
from src.io.load_config import load_config
sw_config = load_config()['space_weather']


# load T2 dataset dir
T2_dir = sw_config['transform']['k_index']['T2_output_dir']

Already in parent directory (current working directory: d:\space-weather-project-scrub)


# Self exploration to scrape site location metadata in a structured way

In [2]:
def get_soup_content(url):

    try:
        # Send a GET request to the URL
        response = requests.get(url)

        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            # Access the HTML content using the .text attribute
            html_content = response.text
            #print(html_content)
            soup = BeautifulSoup(html_content, 'html.parser')
            return soup
        else:
            print(f"Failed to retrieve the page. Status code: {response.status_code}")

    except requests.exceptions.RequestException as e:
        # Handle any potential errors during the request (e.g., connection issues)
        print(f"An error occurred: {e}")



In [50]:
url = 'https://sws.bom.gov.au/World_Data_Centre/2/1/7' 

soup = get_soup_content(url)

# Example: get the title of the page
page_title = soup.title.string
print(f"Page Title: {page_title}")

# Example: find all links
# for link in soup.find_all('a'):
#     link_url = link.get('href')
#     print(f'Link URL: {link_url},\n title: {link.get("title")}\n')

# Example: find all tables
tables = soup.find_all('table')
print(tables)

for table in tables:
    # Process each table (e.g., extract rows and cells)
    for row in table.find_all('tr'):
        cells = [cell.text.strip() for cell in row.find_all(['td', 'th'])]
        # Do something with the cell data (e.g., store in a list or print)
        print(cells)



Page Title: SWS - Data Catalogue - Alice Springs
[<table>
<tr class="hidden"><th>Item</th><th>Value</th></tr>
<tr><td>Station Name</td><td>Alice Springs</td> </tr>
<tr><td>Alternative Name</td><td>None</td> </tr>
</table>, <table>
<tr class="hidden"><th>Item</th><th>Value</th></tr>
<tr><td>Geographic</td><td>Lat. -23.81 Long. 133.90E</td> </tr>
<tr><td>Geomagnetic</td><td>Lat. -34.44 Long. 206.95</td> </tr>
<tr><td>Dip</td><td>-34.3</td> </tr>
<tr><td>Split (0.5*gyro frequency) </td><td>0.6</td> </tr>
<tr><td>L Shell</td><td>1.47</td> </tr>
<tr><td>Time Noon</td><td>03 UT</td> </tr>
</table>, <table>
<tr class="hidden"><th>Item</th><th>Value</th></tr>
<tr><td>Years of Data</td><td> 6 months</td> </tr>
<tr><td>Opened</td><td>May 1985</td> </tr>
<tr><td>Closed</td><td>November 1985</td> </tr>
<tr><td>Status</td><td>Closed</td> </tr>
</table>, <table>
<tr class="hidden"><th>Item</th><th>Value</th></tr>
<tr><td>IPS Code</td><td>3357</td> </tr>
<tr><td>URSI Code</td><td>AE42L</td> </tr>
<tr

```
<table>
<tr class="hidden"><th>Item</th><th>Value</th></tr>
<tr><td>Geographic</td><td>Lat. -12.45 Long. 130.95E</td></tr>
<tr><td>Geomagnetic</td><td>Lat. -21.96 Long. 202.84</td></tr>
<tr><td>Dip</td><td>-44.5</td></tr>
<tr><td>Split (0.5*gyro frequency) </td><td>0.6 MHz</td></tr>
<tr><td>L Shell</td><td>1.16</td></tr>
<tr><td>Time Noon</td><td>03 UT</td></tr>
</table>
```
Would correspond to the following table:



| Item | Value |
| :--- | :--- |
| **Geographic** | Lat. -12.45 Long. 130.95E |
| **Geomagnetic** | Lat. -21.96 Long. 202.84 |
| **Dip** | -44.5 |
| **Split (0.5 * gyro frequency)** | 0.6 MHz |
| **L Shell** | 1.16 |
| **Time Noon** | 03 UT |

**Each row is enclosed in `<tr></tr>` tags, and each cell is enclosed in `<td></td>` or `<th></th>` tags.**
- tr = table row
- td = table data
- th = table header

In [31]:
# the URL for the map page containing hyperlinks for each station's metadata
url = 'https://sws.bom.gov.au/World_Data_Centre/2/1/1'
get_soup_content(url).find_all('area')



[<area alt="Mawson" coords="0,465,40,535" href="/World_Data_Centre/2/1/23" shape="rect"/>,
 <area alt="Davis" coords="50,468,80,533" href="/World_Data_Centre/2/1/22" shape="rect"/>,
 <area alt="Casey" coords="90,467,125,535" href="/World_Data_Centre/2/1/24" shape="rect"/>,
 <area alt="Macquarie Island" coords="391,463,465,532" href="/World_Data_Centre/2/1/26" shape="rect"/>,
 <area alt="Scott Base" coords="583,467,642,531" href="/World_Data_Centre/2/1/25" shape="rect"/>,
 <area alt="Learmonth" coords="65,226,139,248" href="/World_Data_Centre/2/1/21" shape="rect"/>,
 <area alt="Culgoora" coords="342,311,396,330" href="/World_Data_Centre/2/1/27" shape="rect"/>,
 <area alt="Cocos Islands" coords="0,98,70,120" href="/World_Data_Centre/2/1/20" shape="rect"/>,
 <area alt="Niue" coords="666,170,707,194" href="/World_Data_Centre/2/1/18" shape="rect"/>,
 <area alt="Norfolk Island" coords="506,293,580,314" href="/World_Data_Centre/2/1/19" shape="rect"/>,
 <area alt="Vanimo" coords="309,12,371,37

In [35]:
# raw HTML string parsed by BeautifulSoup
get_soup_content(url)

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">

<html lang="en-AU">
<head>
<!-- Google tag (gtag.js) -->
<script async="" src="https://www.googletagmanager.com/gtag/js?id=G-7N5HE7LPBD"></script>
<script>
  window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());

  gtag('config', 'G-7N5HE7LPBD');
</script>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="width=device-width,initial-scale=1.0" name="viewport"/>
<meta content="http://www.sws.bom.gov.au/" name="DC.Identifier" scheme="URI"/>
<meta content="c=AU; co=Commonwealth of Australia; ou=Department of Sustainability, Environment, Water, Population and Communities; ou=Central Office; ou=Bureau of Meteorology; ou=Space Weather Services" name="DC.Creator" scheme="GOLD"/>
<meta content="c=AU; co=Commonwealth of Australia; ou=Department of Sustainability, Environment, Water, Population and Co

In [23]:
# simply parsing the stations and their respective relative hyperlinks 
station_href_elements = {e.get('alt'):e.get('href') for e in get_soup_content(url).find_all('area')}
station_href_elements

{'Mawson': '/World_Data_Centre/2/1/23',
 'Davis': '/World_Data_Centre/2/1/22',
 'Casey': '/World_Data_Centre/2/1/24',
 'Macquarie Island': '/World_Data_Centre/2/1/26',
 'Scott Base': '/World_Data_Centre/2/1/25',
 'Learmonth': '/World_Data_Centre/2/1/21',
 'Culgoora': '/World_Data_Centre/2/1/27',
 'Cocos Islands': '/World_Data_Centre/2/1/20',
 'Niue': '/World_Data_Centre/2/1/18',
 'Norfolk Island': '/World_Data_Centre/2/1/19',
 'Vanimo': '/World_Data_Centre/2/1/2',
 'Darwin': '/World_Data_Centre/2/1/3',
 'Port Moresby': '/World_Data_Centre/2/1/4',
 'Townsville': '/World_Data_Centre/2/1/5',
 'Tennant Creek': '/World_Data_Centre/2/1/6',
 'Alice Springs': '/World_Data_Centre/2/1/7',
 'Watheroo': '/World_Data_Centre/2/1/8',
 'Mundaring': '/World_Data_Centre/2/1/9',
 'Brisbane': '/World_Data_Centre/2/1/10',
 'Woomera': '/World_Data_Centre/2/1/11',
 'Salisbury': '/World_Data_Centre/2/1/12',
 'Camden': '/World_Data_Centre/2/1/13',
 'Canberra': '/World_Data_Centre/2/1/14',
 'Hobart': '/World_Da

In [24]:
def get_station_metadata(base_url, href):
    
    url_station = base_url + href
    soup = get_soup_content(url_station)

    tables = soup.find_all('table')
    print(f'len(tables): {len(tables)}')

    for table in tables:
        # Process each table (e.g., extract rows and cells)
        for row in table.find_all('tr'): # tr = table row
            # td = table data, th = table header
            cells = [cell.text.strip() for cell in row.find_all(['td', 'th'])]
            # Do something with the cell data (e.g., store in a list or print)
            print(cells)
        print()


# in the space weather API get-k-index endpoint, "Narrabri" is used instead of "Culgoora"
base_url = 'https://sws.bom.gov.au'
get_station_metadata(base_url, station_href_elements['Auckland'])

# create json dict of `{station_name: {alternative_name:.., lat_raw:.., long_raw:.., geometry:.., source_url:base_url+href, retrieved_at_utc:..}}`
# so that when joining station location to T2, for each row in T2, check if its location exists in the metadata (station_name and alternative_name), but 
# for alternative_name I think we should not use exact matching, just check whether the location is in the Alternative Name entry or not, as per the following output
# if it exsits then parse the coordinates e.g. ['Geographic', 'Lat. -30.28 Long. 149.58E'] into geometry_raw e.g. 'Lat. -30.28 Long. 149.58E', lat, long, geometry.

# but to determine the format of the geo coordinates, we probably shoould gather them from all locations (e.g. long has 'E' indicator but not the latitude)

len(tables): 5
['Item', 'Value']
['Station Name', 'Auckland']
['Alternative Name', 'None']

['Item', 'Value']
['Geographic', 'Lat. -37.00 Long. 175.00E']
['Geomagnetic', 'Lat. -42.88 Long. 256.11']
['Dip', '-62.90']
['Split (0.5*gyro frequency)', '0.8']
['L Shell', '1.86']
['Time Noon', '00 UT']

['Item', 'Value']
['Years of Data', '31 years']
['Opened', 'December 1967']
['Closed', 'March 1998']
['Status', 'Closed']

['Item', 'Value']
['IPS Code', '4464']
['URSI Code', 'AU63P']
['WDC Code', '9063P']
['ESSA Code', '63P']

['Item', 'Value']
['Ionogram', 'December 1967-August 1998, Digital']



In [ ]:
def get_station_coord(base_url, href):
    
    url_station = base_url + href
    soup = get_soup_content(url_station)

    tables = soup.find_all('table')

    for table in tables:
        # Process each table (e.g., extract rows and cells)
        for row in table.find_all('tr'): # tr = table row
            # td = table data, th = table header
            cells = [cell.text.strip() for cell in row.find_all(['td', 'th'])]
            # Do something with the cell data (e.g., store in a list or print)
            if cells[0] == 'Geographic':
                return cells[1]
            #print(cells)
        #print()


station_coords = {e.get('alt'):get_station_coord(base_url, e.get('href')) for e in get_soup_content(url).find_all('area')}

In [15]:
station_coords

{'Mawson': 'Lat. -67.60 Long. 62.88E',
 'Davis': 'Lat. -68.58 Long. 77.96E',
 'Casey': 'Lat. -66.30 Long. 110.50E',
 'Macquarie Island': 'Lat. -54.50 Long. 158.95E',
 'Scott Base': 'Lat. -77.80 Long. 166.80E',
 'Learmonth': 'Lat. -22.25 Long. 114.08E',
 'Culgoora': 'Lat. -30.28 Long. 149.58E',
 'Cocos Islands': 'Lat. -12.20 Long. 96.80E',
 'Niue': 'Lat. -19.07 Long. 190.07E',
 'Norfolk Island': 'Lat. -29.03 Long. 167.97E',
 'Vanimo': 'Lat. -2.70 Long. 141.30E',
 'Darwin': 'Lat. -12.45 Long. 130.95E',
 'Port Moresby': 'Lat. -9.40 Long. 147.10E',
 'Townsville': 'Lat. -19.63 Long. 146.85E',
 'Tennant Creek': 'Lat. -19.65 Long. 134.25E',
 'Alice Springs': 'Lat. -23.81 Long. 133.90E',
 'Watheroo': 'Lat. -30.30 Long. 115.90E',
 'Mundaring': 'Lat. -31.98 Long. 116.22E',
 'Brisbane': 'Lat. -27.53 Long. 152.92E',
 'Woomera': 'Lat. -30.80 Long. 136.30E',
 'Salisbury': 'Lat. -34.70 Long. 138.60E',
 'Camden': 'Lat. -34.05 Long. 150.67',
 'Canberra': 'Lat. -35.32 Long. 149.00E',
 'Hobart': 'Lat. -4

# Developed code to parse station locations into a table

In [2]:
import re
from datetime import datetime, timezone
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


def get_soup_content(url: str, timeout: int = 30) -> BeautifulSoup:
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def extract_key_value_rows(soup: BeautifulSoup) -> dict[str, str]:
    """
    Extract all 2-column table rows into a flat dict.
    Example row:
        ['Geographic', 'Lat. -30.28 Long. 149.58E']
    """
    values = {}

    for table in soup.find_all("table"):
        for row in table.find_all("tr"):
            cells = [cell.get_text(strip=True) for cell in row.find_all(["td", "th"])]
            if len(cells) == 2 and cells[0] != "Item":
                values[cells[0]] = cells[1]

    return values


def parse_alternative_names(alternative_name_raw: str | None) -> list[str]:
    """
    Tentative rule:
    'Narrabri (although not used with data reports)' -> ['Narrabri']

    Keep this conservative:
    - preserve the raw string separately
    - only derive a leading alias before any parenthetical note
    """
    if not alternative_name_raw:
        return []

    candidate = re.sub(r"\s*\(.*?\)\s*", "", alternative_name_raw).strip()

    if not candidate:
        return []

    return [candidate]


def parse_geographic(geometry_raw: str | None) -> tuple[float | None, float | None, str | None]:
    """
    Parse strings like:
        'Lat. -30.28 Long. 149.58E'
        'Lat. -12.45 Long. 130.95E'

    Returns:
        (lat, lon, geometry_wkt)
    """
    if not geometry_raw:
        return None, None, None

    match = re.search(
        r"Lat\.\s*([+-]?\d+(?:\.\d+)?)\s+Long\.\s*([+-]?\d+(?:\.\d+)?)([EW])?",
        geometry_raw
    )
    if not match:
        return None, None, None

    lat = float(match.group(1))
    lon = float(match.group(2))
    lon_dir = match.group(3)

    if lon_dir == "W":
        lon = -abs(lon)
    elif lon_dir == "E":
        lon = abs(lon)

    geometry = f"POINT ({lon} {lat})"
    return lat, lon, geometry


def extract_station_metadata(
    base_url: str = "https://sws.bom.gov.au",
    map_page_url: str = "https://sws.bom.gov.au/World_Data_Centre/2/1/1",
) -> pd.DataFrame:
    
    """
    Build one canonical metadata row per station from the WDC map page.
    """

    retrieved_at_utc = datetime.now(timezone.utc).isoformat()

    # scrape station `href` links from the WDC map page 
    map_soup = get_soup_content(map_page_url)

    print("Scraping all <area> elements in the World Data Center map page..")
    area_elements = map_soup.find_all("area")
    num_areas = len(area_elements)
    print(f"{num_areas} sites found.")

    rows = []

    for i, area in enumerate(area_elements):
        
        station_display_name = area.get("alt")
        href = area.get("href")

        print(f"Obtaining metadata for station {station_display_name} ({i+1}/{num_areas})")

        if not station_display_name or not href:
            print(f"Station display name does not exist or link does not exist. Continuing to the next station...")
            continue

        # visit each defail page (base_url + corresponding `href`)
        source_url = urljoin(base_url, href)
        print(f"Visiting detail page for station {station_display_name}")
        station_soup = get_soup_content(source_url)

        print("Extracting metadata..")

        kv = extract_key_value_rows(station_soup)

        # extract `Station Name`, `Alternative Name` and `Geographic`
        station_name = kv.get("Station Name", station_display_name)

        # map page explicitly states non existent alternative names as "None"
        # for our purposes we parse it instead to a NoneType 
        alternative_name_raw = kv.get("Alternative Name") if kv.get("Alternative Name") != "None" else None
        alternative_names = parse_alternative_names(alternative_name_raw) if alternative_name_raw else None

        geometry_raw = kv.get("Geographic")

        # store station metadata as a canonical row
        lat, lon, geometry = parse_geographic(geometry_raw)

        rows.append(
            {
                "station_name": station_name,
                "alternative_name_raw": alternative_name_raw,
                "alternative_names": alternative_names,
                "geometry_raw": geometry_raw,
                "lat": lat,
                "long": lon,
                "geometry_txt": geometry,
                "source_url": source_url,
                "retrieved_at_utc": retrieved_at_utc,
            }
        )

    return pd.DataFrame(rows)


In [3]:
station_metadata = extract_station_metadata()
station_metadata


Scraping all <area> elements in the World Data Center map page..
31 sites found.
Obtaining metadata for station Mawson (1/31)
Visiting detail page for station Mawson
Extracting metadata..
Obtaining metadata for station Davis (2/31)
Visiting detail page for station Davis
Extracting metadata..
Obtaining metadata for station Casey (3/31)
Visiting detail page for station Casey
Extracting metadata..
Obtaining metadata for station Macquarie Island (4/31)
Visiting detail page for station Macquarie Island
Extracting metadata..
Obtaining metadata for station Scott Base (5/31)
Visiting detail page for station Scott Base
Extracting metadata..
Obtaining metadata for station Learmonth (6/31)
Visiting detail page for station Learmonth
Extracting metadata..
Obtaining metadata for station Culgoora (7/31)
Visiting detail page for station Culgoora
Extracting metadata..
Obtaining metadata for station Cocos Islands (8/31)
Visiting detail page for station Cocos Islands
Extracting metadata..
Obtaining metad

,station_name,alternative_name_raw,alternative_names,geometry_raw,lat,long,geometry_txt,source_url,retrieved_at_utc
0,Mawson,None,None,Lat. -67.60 Long. 62.88E,-67.60,62.88,POINT (62.88 -67.6),https://sws.bom.gov.au/World_Data_Centre/2/1/23,2026-04-16T06:47:28.506502+00:00
1,Davis,None,None,Lat. -68.58 Long. 77.96E,-68.58,77.96,POINT (77.96 -68.58),https://sws.bom.gov.au/World_Data_Centre/2/1/22,2026-04-16T06:47:28.506502+00:00
2,Casey,Wilkes,[Wilkes],Lat. -66.30 Long. 110.50E,-66.30,110.50,POINT (110.5 -66.3),https://sws.bom.gov.au/World_Data_Centre/2/1/24,2026-04-16T06:47:28.506502+00:00
3,Macquarie Island,None,None,Lat. -54.50 Long. 158.95E,-54.50,158.95,POINT (158.95 -54.5),https://sws.bom.gov.au/World_Data_Centre/2/1/26,2026-04-16T06:47:28.506502+00:00
4,Scott Base,None,None,Lat. -77.80 Long. 166.80E,-77.80,166.80,POINT (166.8 -77.8),https://sws.bom.gov.au/World_Data_Centre/2/1/25,2026-04-16T06:47:28.506502+00:00
5,Learmonth,None,None,Lat. -22.25 Long. 114.08E,-22.25,114.08,POINT (114.08 -22.25),https://sws.bom.gov.au/World_Data_Centre/2/1/21,2026-04-16T06:47:28.506502+00:00
6,Culgoora,Narrabri (although not used with data reports),[Narrabri],Lat. -30.28 Long. 149.58E,-30.28,149.58,POINT (149.58 -30.28),https://sws.bom.gov.au/World_Data_Centre/2/1/27,2026-04-16T06:47:28.506502+00:00
7,Cocos Islands,Keeling Islands,[Keeling Islands],Lat. -12.20 Long. 96.80E,-12.20,96.80,POINT (96.8 -12.2),https://sws.bom.gov.au/World_Data_Centre/2/1/20,2026-04-16T06:47:28.506502+00:00
8,Niue,None,None,Lat. -19.07 Long. 190.07E,-19.07,190.07,POINT (190.07 -19.07),https://sws.bom.gov.au/World_Data_Centre/2/1/18,2026-04-16T06:47:28.506502+00:00
9,Norfolk Island,None,None,Lat. -29.03 Long. 167.97E,-29.03,167.97,POINT (167.97 -29.03),https://sws.bom.gov.au/World_Data_Centre/2/1/19,2026-04-16T06:47:28.506502+00:00


In [27]:
station_metadata.loc[
    station_metadata["alternative_name_raw"].notnull(),
    ["station_name", "alternative_name_raw", "alternative_names"]
]

,station_name,alternative_name_raw,alternative_names
2,Casey,Wilkes,[Wilkes]
6,Culgoora,Narrabri (although not used with data reports),[Narrabri]
7,Cocos Islands,Keeling Islands,[Keeling Islands]
11,Darwin,none (although there are several - ASWFC and D...,[none]
16,Watheroo,Mundaring - nearby station,[Mundaring - nearby station]
17,Mundaring,"Watheroo (an earlier, nearby site)",[Watheroo]
18,Brisbane,Bribie Island,[Bribie Island]
21,Camden,Sydney,[Sydney]
25,Christchurch,Godley Head (actually a different nearby locat...,[Godley Head]
27,Perth,Bickley,[Bickley]


# Version 2 code for site metadata, after rectifying the spec

- Note we decided to NOT parse `alternative_names` because we already used a lookup table.

In [ ]:
import re
import warnings
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# Spec section 2:
# These are the source table keys the metadata builder is expected to extract.
SITE_METADATA_SOURCE_KEYS = {"Station Name", "Alternative Name", "Geographic"}



def get_soup_content(url: str, timeout: int = 30) -> BeautifulSoup:
    """
    Fetch a web page and parse its HTML into a BeautifulSoup object.

    Spec relevance:
    - `src/metadata/site_location.py::get_soup_content`
    - Used for both the WDC map page and each station detail page.

    Notes:
    - HTTP/network failures are not swallowed here. In a future entrypoint, the
      standard logging wrapper can record the stack trace.
    """
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def extract_key_value_rows(soup: BeautifulSoup) -> dict[str, str | None]:
    """
    Extract station-page 2-column table rows into a flat dict.

    Spec relevance:
    - Extract values that come immediately after `Station Name`,
      `Alternative Name`, and `Geographic`.
    - Failure mode: if a relevant row is jagged, e.g. only `['Geographic']`,
      warn and store null rather than failing fast.
    """
    values = {}

    for table in soup.find_all("table"):
        for row in table.find_all("tr"):
            cells = [cell.get_text(strip=True) for cell in row.find_all(["td", "th"])]

            if not cells:
                continue

            key = cells[0]

            # Ignore the repeated table header row: ['Item', 'Value'].
            if key == "Item":
                continue

            # Spec section 4:
            # Jagged table rows should warn, not fail fast.
            if key in SITE_METADATA_SOURCE_KEYS and len(cells) < 2:
                warnings.warn(
                    f"Expected value after '{key}', but parsed row only had: {cells!r}"
                )
                values[key] = None
                continue

            # The station detail pages are expected to be key-value rows.
            if len(cells) >= 2:
                values[key] = cells[1]

    return values


def parse_geographic(geometry_raw: str | None) -> tuple[float | None, float | None, str | None]:
    """
    Parse textual geographic coordinates into latitude, longitude, and WKT-like text.

    Spec relevance:
    - Expected input resembles: 'Lat. -30.28 Long. 149.58E'
    - Output schema fields: `lat`, `lon`, `geometry_txt`
    - Example: 'Lat. -31.54 Long. 159.08E' -> POINT (159.08 -31.54)

    Returns:
    - `(lat, lon, geometry_txt)`
    """
    if geometry_raw is None or pd.isna(geometry_raw):
        return None, None, None

    text = str(geometry_raw).strip()

    match = re.search(
        r"Lat\.\s*([+-]?\d+(?:\.\d+)?)\s+Long\.\s*([+-]?\d+(?:\.\d+)?)([EW])?",
        text,
    )

    if not match:
        # inconsistent with spec: should fail fast?
        warnings.warn(f"Could not parse Geographic value: {geometry_raw!r}")
        return None, None, None

    lat = float(match.group(1))
    lon = float(match.group(2))
    lon_direction = match.group(3)

    # Current observed WDC values mostly use E; this keeps W values correct too.
    if lon_direction == "W":
        lon = -abs(lon)
    elif lon_direction == "E":
        lon = abs(lon)

    geometry_txt = f"POINT ({lon} {lat})"
    return lat, lon, geometry_txt


def extract_station_metadata(
    base_url: str = "https://sws.bom.gov.au",
    map_page_url: str = "https://sws.bom.gov.au/World_Data_Centre/2/1/1",
    metadata_file_path: str = "notebooks/site-metadata.parquet",
) -> str:
    """
    Build one canonical station metadata row per WDC station and save as Parquet.

    Spec relevance:
    - Scrape station `<href>` links from the WDC map page.
    - Visit each station detail page using `base_url + href`.
    - Extract `Station Name`, `Alternative Name`, and `Geographic`.
    - Parse:
      - `Station Name` -> `station_name`
      - `Alternative Name` -> `alternative_name_raw`
      - `Geographic` -> `geometry_raw`, `lat`, `lon`, `geometry_txt`
    - Add `source_url` and `retrieved_at_utc`.
    - Materialize the station metadata as a Parquet table.

    Returns:
    - The output Parquet file path as a string.
    """
    retrieved_at_utc = datetime.now(timezone.utc).isoformat()

    # Spec section 2.1:
    # Scrape station href links from the WDC map page.
    map_soup = get_soup_content(map_page_url)
    area_elements = map_soup.find_all("area")
    num_area_elements = len(area_elements)

    rows = []

    for i, area in enumerate(area_elements):
        station_display_name = area.get("alt")
        print(f"Parsing {i+1} out of {num_area_elements} stations..")
        href = area.get("href")

        # Defensive guard: malformed map entries are skipped with a warning.
        if not station_display_name or not href:
            warnings.warn(
                f"Skipping map area because `alt` or `href` is missing: {area!r}"
            )
            continue

        # Spec section 2.2:
        # Visit the station detail page.
        source_url = urljoin(base_url, href)
        station_soup = get_soup_content(source_url)

        # Spec section 2.3:
        # Extract source fields from station metadata tables.
        kv = extract_key_value_rows(station_soup)

        station_name = kv.get("Station Name")
        alternative_name_raw = kv.get("Alternative Name")
        geometry_raw = kv.get("Geographic")

        # Spec section 4:
        # Missing key fields warn but do not fail fast.
        if station_name is None:
            warnings.warn(
                f"`Station Name` missing for {source_url}; using map display name "
                f"{station_display_name!r} as fallback."
            )
            station_name = station_display_name

        if alternative_name_raw is None:
            warnings.warn(f"`Alternative Name` missing for station {station_name!r}.")

        if geometry_raw is None:
            warnings.warn(f"`Geographic` missing for station {station_name!r}.")

        # Parse `Geographic` into `lat`, `lon`, and human-readable `geometry_txt`.
        lat, lon, geometry_txt = parse_geographic(geometry_raw)

        # Store one canonical row per station.
        rows.append(
            {
                "station_name": station_name,
                "alternative_name_raw": alternative_name_raw,
                "geometry_raw": geometry_raw,
                "lat": lat,
                "lon": lon,
                "geometry_txt": geometry_txt,
                "source_url": source_url,
                "retrieved_at_utc": retrieved_at_utc,
            }
        )

    station_metadata = pd.DataFrame(
        rows,
        columns=[
            "station_name",
            "alternative_name_raw",
            "geometry_raw",
            "lat",
            "lon",
            "geometry_txt",
            "source_url",
            "retrieved_at_utc",
        ],
    )

    output_path = Path(metadata_file_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Spec section 2:
    # Materialize station metadata as Parquet.
    station_metadata.to_parquet(output_path, index=False)

    return str(output_path)


# Try saving site metadata at `notebooks/` first

In [23]:
metadata_path = extract_station_metadata(
    metadata_file_path="notebooks/site-metadata.parquet"
)


station_metadata = pd.read_parquet(metadata_path)

station_metadata


Parsing 1 out of 31 stations..
Parsing 2 out of 31 stations..
Parsing 3 out of 31 stations..
Parsing 4 out of 31 stations..
Parsing 5 out of 31 stations..
Parsing 6 out of 31 stations..
Parsing 7 out of 31 stations..
Parsing 8 out of 31 stations..
Parsing 9 out of 31 stations..
Parsing 10 out of 31 stations..
Parsing 11 out of 31 stations..
Parsing 12 out of 31 stations..
Parsing 13 out of 31 stations..
Parsing 14 out of 31 stations..
Parsing 15 out of 31 stations..
Parsing 16 out of 31 stations..
Parsing 17 out of 31 stations..
Parsing 18 out of 31 stations..
Parsing 19 out of 31 stations..
Parsing 20 out of 31 stations..
Parsing 21 out of 31 stations..
Parsing 22 out of 31 stations..
Parsing 23 out of 31 stations..
Parsing 24 out of 31 stations..
Parsing 25 out of 31 stations..
Parsing 26 out of 31 stations..
Parsing 27 out of 31 stations..
Parsing 28 out of 31 stations..
Parsing 29 out of 31 stations..
Parsing 30 out of 31 stations..
Parsing 31 out of 31 stations..


,station_name,alternative_name_raw,geometry_raw,lat,lon,geometry_txt,source_url,retrieved_at_utc
0,Mawson,None,Lat. -67.60 Long. 62.88E,-67.60,62.88,POINT (62.88 -67.6),https://sws.bom.gov.au/World_Data_Centre/2/1/23,2026-04-30T06:58:11.575704+00:00
1,Davis,None,Lat. -68.58 Long. 77.96E,-68.58,77.96,POINT (77.96 -68.58),https://sws.bom.gov.au/World_Data_Centre/2/1/22,2026-04-30T06:58:11.575704+00:00
2,Casey,Wilkes,Lat. -66.30 Long. 110.50E,-66.30,110.50,POINT (110.5 -66.3),https://sws.bom.gov.au/World_Data_Centre/2/1/24,2026-04-30T06:58:11.575704+00:00
3,Macquarie Island,None,Lat. -54.50 Long. 158.95E,-54.50,158.95,POINT (158.95 -54.5),https://sws.bom.gov.au/World_Data_Centre/2/1/26,2026-04-30T06:58:11.575704+00:00
4,Scott Base,None,Lat. -77.80 Long. 166.80E,-77.80,166.80,POINT (166.8 -77.8),https://sws.bom.gov.au/World_Data_Centre/2/1/25,2026-04-30T06:58:11.575704+00:00
5,Learmonth,None,Lat. -22.25 Long. 114.08E,-22.25,114.08,POINT (114.08 -22.25),https://sws.bom.gov.au/World_Data_Centre/2/1/21,2026-04-30T06:58:11.575704+00:00
6,Culgoora,Narrabri (although not used with data reports),Lat. -30.28 Long. 149.58E,-30.28,149.58,POINT (149.58 -30.28),https://sws.bom.gov.au/World_Data_Centre/2/1/27,2026-04-30T06:58:11.575704+00:00
7,Cocos Islands,Keeling Islands,Lat. -12.20 Long. 96.80E,-12.20,96.80,POINT (96.8 -12.2),https://sws.bom.gov.au/World_Data_Centre/2/1/20,2026-04-30T06:58:11.575704+00:00
8,Niue,None,Lat. -19.07 Long. 190.07E,-19.07,190.07,POINT (190.07 -19.07),https://sws.bom.gov.au/World_Data_Centre/2/1/18,2026-04-30T06:58:11.575704+00:00
9,Norfolk Island,None,Lat. -29.03 Long. 167.97E,-29.03,167.97,POINT (167.97 -29.03),https://sws.bom.gov.au/World_Data_Centre/2/1/19,2026-04-30T06:58:11.575704+00:00


# Join logic (lookup table)

## Hardcode lookup table

In [26]:
import pandas as pd

location_lookup = pd.DataFrame(
    [
        {"api_location": "Alice Springs", "canonical_station_name": "Alice Springs"},
        {"api_location": "Canberra", "canonical_station_name": "Canberra"},
        {"api_location": "Cocos Island", "canonical_station_name": "Cocos Islands"},
        {"api_location": "Narrabri", "canonical_station_name": "Culgoora"},
        {"api_location": "Darwin", "canonical_station_name": "Darwin"},
        {"api_location": "Hobart", "canonical_station_name": "Hobart"},
        {"api_location": "Launceston", "canonical_station_name": "Launceston"},
        {"api_location": "Learmonth", "canonical_station_name": "Learmonth"},
        {"api_location": "Melbourne", "canonical_station_name": None},
        {"api_location": "Norfolk Island", "canonical_station_name": "Norfolk Island"},
        {"api_location": "Perth", "canonical_station_name": "Perth"},
        {"api_location": "Sydney", "canonical_station_name": "Camden"},
        {"api_location": "Townsville", "canonical_station_name": "Townsville"},
        {"api_location": "Casey", "canonical_station_name": "Casey"},
        {"api_location": "Davis", "canonical_station_name": "Davis"},
        {"api_location": "Macquarie Island", "canonical_station_name": "Macquarie Island"},
        {"api_location": "Mawson", "canonical_station_name": "Mawson"},
        {"api_location": "Australian region", "canonical_station_name": None},
    ]
)

location_lookup


,api_location,canonical_station_name
0,Alice Springs,Alice Springs
1,Canberra,Canberra
2,Cocos Island,Cocos Islands
3,Narrabri,Culgoora
4,Darwin,Darwin
5,Hobart,Hobart
6,Launceston,Launceston
7,Learmonth,Learmonth
8,Melbourne,None
9,Norfolk Island,Norfolk Island


In [ ]:
# brief check on current T2
t2 = duckdb.read_parquet(T2_dir).df()
t2

,location,valid_time,kindex,flag
0,Australian region,2020-09-08 18:00:00,1,False
1,Australian region,2020-09-12 00:00:00,1,False
2,Australian region,2020-09-14 18:00:00,0,False
3,Australian region,2020-09-19 18:00:00,0,False
4,Australian region,2020-09-20 06:00:00,0,False
...,...,...,...,...
31628,Australian region,2022-11-03 09:00:00,5,False
31629,Australian region,2022-11-07 03:00:00,2,False
31630,Australian region,2022-11-08 06:00:00,2,False
31631,Australian region,2022-11-08 18:00:00,1,False


In [27]:
import duckdb

T2_path = "data/02-preprocessed/space_weather/k_index/T2/**/*.parquet"
site_metadata_path = "notebooks/site-metadata.parquet"

duckdb.register("location_lookup", location_lookup)

query = f"""
WITH t2 AS (
    SELECT *
    FROM read_parquet('{T2_path}')
),

metadata AS (
    SELECT
        station_name,
        alternative_name_raw,
        geometry_raw,
        lat,
        lon,
        geometry_txt,
        source_url,
        retrieved_at_utc
    FROM read_parquet('{site_metadata_path}')
),

joined AS (
    SELECT
        t2.location,
        t2.valid_time,
        t2.kindex,
        t2.flag,
        l.canonical_station_name,
        m.station_name,
        m.alternative_name_raw,
        m.geometry_raw,
        m.lat,
        m.lon,
        m.geometry_txt,
        m.source_url,
        m.retrieved_at_utc,
        CASE
            WHEN l.api_location IS NULL THEN 'unmapped_api_location'
            WHEN l.canonical_station_name IS NULL THEN 'known_without_station'
            WHEN m.station_name IS NULL THEN 'lookup_target_missing_in_metadata'
            ELSE 'lookup_match'
        END AS match_type
    FROM t2
    LEFT JOIN location_lookup AS l
        ON t2.location = l.api_location
    LEFT JOIN metadata AS m
        ON l.canonical_station_name = m.station_name
)

SELECT *
FROM joined
"""

t2_with_metadata = duckdb.sql(query).df()


In [28]:
# view all matched and unmatched locations
t2_with_metadata[
    ["location", "canonical_station_name", "station_name", "geometry_txt", "match_type"]
].drop_duplicates().sort_values("location")


,location,canonical_station_name,station_name,geometry_txt,match_type
131,Alice Springs,Alice Springs,Alice Springs,POINT (133.9 -23.81),lookup_match
478,Australian region,None,None,None,known_without_station
296,Cocos Island,Cocos Islands,Cocos Islands,POINT (96.8 -12.2),lookup_match
0,Darwin,Darwin,Darwin,POINT (130.95 -12.45),lookup_match


In [29]:
t2_with_metadata["match_type"].value_counts(dropna=False)

match_type
known_without_station    23617
lookup_match              8016
Name: count, dtype: int64

In [30]:
t2_with_metadata.loc[
    t2_with_metadata["match_type"] != "lookup_match",
    ["location", "canonical_station_name", "match_type"]
].drop_duplicates().sort_values("location")


,location,canonical_station_name,match_type
478,Australian region,None,known_without_station
